In [18]:
import pandas as pd
import numpy as np
import re
import torch
from typing import Dict, List, Tuple, Any
from collections import defaultdict

# Transformers historiques
from transformers import AutoTokenizer, AutoModel
from sentence_transformers import SentenceTransformer, util

In [19]:
print("Loading historical transformer models...")

HISTORICAL_BERT = "dbmdz/bert-base-historic-english-cased"

try:
    print(f"Loading {HISTORICAL_BERT}...")
    hist_tokenizer = AutoTokenizer.from_pretrained(HISTORICAL_BERT)
    hist_model = AutoModel.from_pretrained(HISTORICAL_BERT)
    USE_HISTORICAL = True
    print("Historical BERT loaded successfully")
except Exception as e:
    print(f"Could not load historical model: {e}")
    print(f"Falling back to roberta-base...")
    hist_tokenizer = AutoTokenizer.from_pretrained("roberta-base")
    hist_model = AutoModel.from_pretrained("roberta-base")
    USE_HISTORICAL = False

if USE_HISTORICAL:
    embedder = SentenceTransformer("all-mpnet-base-v2")
    print("Using all-mpnet-base-v2 for embeddings")
else:
    embedder = SentenceTransformer("all-MiniLM-L6-v2")
    print("Using all-MiniLM-L6-v2 for embeddings")

Loading historical transformer models...
Loading dbmdz/bert-base-historic-english-cased...


Some weights of BertModel were not initialized from the model checkpoint at dbmdz/bert-base-historic-english-cased and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Historical BERT loaded successfully
Using all-mpnet-base-v2 for embeddings


In [20]:
class HistoricalTextCleaner:
    def __init__(self):
        self.date_patterns = [
            (r'\b(\d+)\s*(BC|BCE|B\.C\.|B\.C\.E\.)\b', r'\1 BCE'),
            (r'\b(\d+)\s*(AD|CE|A\.D\.|C\.E\.)\b', r'\1 CE'),
            (r'\bcirca\s*(\d+)\s*(?:BCE?|CE)?\b', r'c \1'),
            (r'\b(\d+)\s*-\s*(\d+)\s*(?:century|centuries)\s*(BCE?)?\b', self._format_century_range),
            (r'\b(\d+)(?:st|nd|rd|th)\s+century\s*(BCE?)?\b', self._century_to_years),
            (r'\b(\d+)\s*/\s*(\d+)\s*(?:BCE?|CE)?\b', r'\1-\2'),
        ]
        
        self.historical_names = {
            'hannibal': ['hannibal barca', 'barca', 'hannibal of carthage'],
            'scipio': ['scipio africanus', 'publius cornelius scipio'],
            'caesar': ['julius caesar', 'gaius julius caesar'],
            'bourguiba': ['habib bourguiba'],
            'dido': ['queen dido', 'elissa'],
            'gaiseric': ['genseric', 'king gaiseric'],
            'almahdi': ['abdullah almahdi billah'],
            'ibn khaldun': ['abd alrahman ibn khaldun'],
        }
        
        self.historical_terms = {
            'punic', 'carthaginian', 'roman', 'byzantine', 'vandal', 'arab',
            'ottoman', 'hafsid', 'fatimid', 'aghlamid', 'almohad', 'almoravid',
            'berber', 'numidian', 'phoenician', 'islamic', 'crusader'
        }
    
    def _format_century_range(self, match):
        start, end, era = match.groups()
        era_str = f" {era}" if era else ""
        return f"{start}{end} century{era_str}"
    
    def _century_to_years(self, match):
        century, era = match.groups()
        century_num = int(century)
        start_year = (century_num - 1) * 100 + 1
        end_year = century_num * 100
        era_str = f" {era}" if era else " CE"
        return f"{start_year}{end_year}{era_str}"
    
    def normalize_dates(self, text):
        for pattern, replacement in self.date_patterns:
            if callable(replacement):
                text = re.sub(pattern, replacement, text, flags=re.IGNORECASE)
            else:
                text = re.sub(pattern, replacement, text, flags=re.IGNORECASE)
        return text
    
    def normalize_names(self, text):
        text_lower = text.lower()
        for standard_name, variants in self.historical_names.items():
            for variant in variants:
                if variant in text_lower:
                    pattern = r'\b' + re.escape(variant) + r'\b'
                    text = re.sub(pattern, standard_name, text, flags=re.IGNORECASE)
        return text
    
    def clean_text(self, text):
        if pd.isna(text):
            return ""
        
        text = str(text)
        
        text = self.normalize_dates(text)
        text = self.normalize_names(text)
        text = text.lower()
        
        text = re.sub(r'[^\w\s]', ' ', text)
        text = re.sub(r'\s+', ' ', text)
        text = re.sub(r'[^\w\s]', '', text)
        text = re.sub(r'\s+', ' ', text).strip()
        
        return text
    
    def extract_historical_context(self, text):
        context = {
            'time_period': None,
            'civilization': None,
            'key_figures': []
        }
        
        time_patterns = {
            'ancient': r'\b(ancient|classical|antiquity)\b',
            'medieval': r'\b(medieval|middle ages|dark ages)\b',
            'early modern': r'\b(renaissance|reformation|early modern)\b',
            'modern': r'\b(modern|contemporary|20th century|21st century)\b'
        }
        
        for period, pattern in time_patterns.items():
            if re.search(pattern, text, re.IGNORECASE):
                context['time_period'] = period
                break
        
        civ_patterns = {
            'carthaginian': r'\b(carthaginian|punic)\b',
            'roman': r'\b(roman|rome|republic|empire)\b',
            'islamic': r'\b(islamic|muslim|arab|umayyad|abbasid)\b',
            'ottoman': r'\b(ottoman|turks|sultan)\b',
            'french': r'\b(french|colonial|protectorate)\b'
        }
        
        for civ, pattern in civ_patterns.items():
            if re.search(pattern, text, re.IGNORECASE):
                context['civilization'] = civ
                break
        
        return context

In [21]:
class HistoricalEntityExtractor:
    def __init__(self, tokenizer, model):
        self.tokenizer = tokenizer
        self.model = model
        
    def extract_with_bert(self, text, max_length=512):
        inputs = self.tokenizer(
            text,
            return_tensors='pt',
            truncation=True,
            padding=True,
            max_length=max_length
        )
        
        with torch.no_grad():
            outputs = self.model(**inputs)
        
        last_hidden_state = outputs.last_hidden_state
        token_embeddings = last_hidden_state[0]
        tokens = self.tokenizer.convert_ids_to_tokens(inputs['input_ids'][0])
        
        entities = []
        current_entity = []
        current_type = None
        
        for i, token in enumerate(tokens):
            if token.startswith('##'):
                current_entity.append(token[2:])
            else:
                if current_entity:
                    entities.append({
                        'text': ''.join(current_entity),
                        'type': current_type or 'UNKNOWN',
                        'embedding': token_embeddings[i-1].cpu().numpy()
                    })
                current_entity = [token]
                current_type = self._guess_entity_type(token)
        
        return entities
    
    def _guess_entity_type(self, token):
        token_lower = token.lower()
        
        if any(name in token_lower for name in ['hannibal', 'scipio', 'caesar', 'bourguiba']):
            return 'PERSON'
        elif any(loc in token_lower for loc in ['carthage', 'rome', 'tunis', 'alps', 'zama']):
            return 'LOCATION'
        elif any(date in token_lower for date in ['bc', 'ad', 'century', 'year']):
            return 'DATE'
        elif any(event in token_lower for event in ['battle', 'war', 'siege', 'treaty']):
            return 'EVENT'
        
        return 'OTHER'
    
    def create_historical_embedding(self, text, method='cls'):
        inputs = self.tokenizer(
            text,
            return_tensors='pt',
            truncation=True,
            padding=True,
            max_length=512
        )
        
        with torch.no_grad():
            outputs = self.model(**inputs)
        
        if method == 'cls':
            embedding = outputs.last_hidden_state[0, 0, :]
        elif method == 'mean':
            embedding = outputs.last_hidden_state[0].mean(dim=0)
        elif method == 'max':
            embedding = outputs.last_hidden_state[0].max(dim=0)[0]
        else:
            embedding = outputs.last_hidden_state[0, 0, :]
        
        return embedding.cpu().numpy()

In [22]:
def load_historical_dataset(filepath):
    try:
        df = pd.read_csv('/kaggle/input/datasets/ayanajlaoui/clean-data/historical_events_cleaned_full.csv', encoding='utf-8')
        print(f"Dataset loaded: {len(df)} rows, {len(df.columns)} columns")
        print(f"Sample columns: {df.columns[:5].tolist()}")
        return df
    except Exception as e:
        print(f"Error loading dataset: {e}")
        return None

def create_historical_embeddings(df, cleaner, extractor, text_column='description'):
    print("Creating historical embeddings...")
    
    embeddings = []
    contexts = []
    
    for idx, row in df.iterrows():
        try:
            if text_column in df.columns:
                text = row[text_column]
            else:
                text = f"{row.get('name_of_incident', '')} {row.get('description', '')}"
            
            cleaned_text = cleaner.clean_text(text)
            
            context = cleaner.extract_historical_context(text)
            
            embedding = extractor.create_historical_embedding(
                cleaned_text[:500],
                method='mean'
            )
            
            st_embedding = embedder.encode(cleaned_text, convert_to_tensor=True).cpu().numpy()
            
            embeddings.append({
                'idx': idx,
                'bert_embedding': embedding,
                'st_embedding': st_embedding,
                'text': cleaned_text[:200],
                'context': context,
                'metadata': {
                    'event_id': row.get('event_id', f'event_{idx}'),
                    'name': row.get('name_of_incident', 'Unknown'),
                    'character': row.get('historical_character', 'Unknown'),
                    'year': row.get('year', 'Unknown')
                }
            })
            
            if (idx + 1) % 50 == 0:
                print(f"Processed {idx + 1}/{len(df)} rows")
                
        except Exception as e:
            print(f"Error processing row {idx}: {e}")
            continue
    
    print(f"Created embeddings for {len(embeddings)} historical events")
    return embeddings

def search_historical_events(query, embeddings, extractor, cleaner, top_k=5):
    cleaned_query = cleaner.clean_text(query)
    
    query_embedding_bert = extractor.create_historical_embedding(
        cleaned_query[:500],
        method='mean'
    )
    
    query_embedding_st = embedder.encode(cleaned_query, convert_to_tensor=True).cpu().numpy()
    
    similarities = []
    
    for emb in embeddings:
        sim_bert = np.dot(query_embedding_bert, emb['bert_embedding']) / (
            np.linalg.norm(query_embedding_bert) * np.linalg.norm(emb['bert_embedding'])
        )
        
        sim_st = np.dot(query_embedding_st, emb['st_embedding']) / (
            np.linalg.norm(query_embedding_st) * np.linalg.norm(emb['st_embedding'])
        )
        
        combined_score = (sim_bert * 0.7) + (sim_st * 0.3)
        
        similarities.append((emb, combined_score))
    
    similarities.sort(key=lambda x: x[1], reverse=True)
    
    return similarities[:top_k]

In [23]:
def main():
    print("=" * 60)
    print("HISTORICAL TRANSFORMER Q&A SYSTEM")
    print(f"Using historical model: {USE_HISTORICAL}")
    print("=" * 60)
    
    cleaner = HistoricalTextCleaner()
    extractor = HistoricalEntityExtractor(hist_tokenizer, hist_model)
    
    df = load_historical_dataset("/kaggle/input/datasets/ayanajlaoui/clean-data/historical_events_cleaned_full.csv")
    if df is None:
        print("Could not load dataset. Exiting.")
        return
    
    embeddings = create_historical_embeddings(df, cleaner, extractor)
    
    while True:
        print("\n" + "-" * 50)
        user_query = input("\nAsk a historical question (or 'quit'): ").strip()
        
        if user_query.lower() in ['quit', 'exit', 'q']:
            print("Exiting historical system.")
            break
        
        print(f"\nQuery: {user_query}")
        
        query_normalized = cleaner.clean_text(user_query)
        print(f"Normalized: {query_normalized}")
        
        print(f"\nExtracting entities with {'historical' if USE_HISTORICAL else 'standard'} BERT...")
        entities = extractor.extract_with_bert(user_query)
        
        if entities:
            print(f"Found {len(entities)} entities:")
            for i, entity in enumerate(entities[:5], 1):
                print(f"  {i}. {entity['text']} ({entity['type']})")
        else:
            print("No entities extracted.")
        
        print(f"\nCreating query embedding...")
        query_embedding = extractor.create_historical_embedding(query_normalized[:500])
        print(f"   Shape: {query_embedding.shape}")
        print(f"   First 5 values: {query_embedding[:5]}")
        
        print(f"\nSearching historical database...")
        results = search_historical_events(user_query, embeddings, extractor, cleaner, top_k=3)
        
        if results:
            print(f"\nFound {len(results)} relevant historical events:")
            
            for i, (emb, score) in enumerate(results, 1):
                meta = emb['metadata']
                context = emb['context']
                
                print(f"\n{i}. {meta['name']}")
                print(f"   Historical Figure: {meta['character']}")
                print(f"   Year: {meta['year']}")
                print(f"   Relevance Score: {score:.3f}")
                
                if context['time_period']:
                    print(f"   Time Period: {context['time_period']}")
                if context['civilization']:
                    print(f"   Civilization: {context['civilization']}")
                
                print(f"   Context: {emb['text'][:150]}...")
        else:
            print("\nNo relevant historical events found.")
        
        print("\n" + "=" * 50)

if __name__ == "__main__":
    main()

HISTORICAL TRANSFORMER Q&A SYSTEM
Using historical model: True
Dataset loaded: 114 rows, 34 columns
Sample columns: ['event_id', 'name_of_incident', 'year', 'country', 'place_name']
Creating historical embeddings...
Processed 50/114 rows
Processed 100/114 rows
Created embeddings for 114 historical events

--------------------------------------------------



Ask a historical question (or 'quit'):  How did Carthage fall?



Query: How did Carthage fall?
Normalized: how did carthage fall

Extracting entities with historical BERT...
Found 6 entities:
  1. [CLS] (OTHER)
  2. [UNK] (OTHER)
  3. did (OTHER)
  4. [UNK] (OTHER)
  5. fall (OTHER)

Creating query embedding...
   Shape: (768,)
   First 5 values: [ 0.4692594  -0.22521688 -9.117548   -0.22283536  0.57184964]

Searching historical database...

Found 3 relevant historical events:

1. Mercenary Revolt Suppression
   Historical Figure: Hamilcar Barca
   Year: -238
   Relevance Score: 0.659
   Civilization: carthaginian
   Context: carthage couldn t pay its mercenaries after losing the first punic war they revolted besieging carthage itself i defeated them through brutal tactics ...

2. Raids on Carthaginian Territory
   Historical Figure: Masinissa
   Year: -150
   Relevance Score: 0.658
   Civilization: carthaginian
   Context: spent my final years raiding carthaginian territory seizing their richest lands carthage couldn t fight back without roman per


Ask a historical question (or 'quit'):  Tell me about the founding of Carthage.



Query: Tell me about the founding of Carthage.
Normalized: tell me about the founding of carthage

Extracting entities with historical BERT...
Found 9 entities:
  1. [CLS] (OTHER)
  2. [UNK] (OTHER)
  3. me (OTHER)
  4. about (OTHER)
  5. the (OTHER)

Creating query embedding...
   Shape: (768,)
   First 5 values: [ 6.8357754e-01 -2.6671284e-01 -9.1570253e+00  2.8327818e-03
  4.2519066e-02]

Searching historical database...

Found 3 relevant historical events:

1. Phoenician Colonial Expansion
   Historical Figure: Dido (Elissa)
   Year: -814
   Relevance Score: 0.759
   Context: founded carthage after fleeing tyre my brother killed my husband for his wealth so i escaped with loyal followers i bought land from the berbers as mu...

2. Sacred Band Formation
   Historical Figure: Carthaginian Army
   Year: -350
   Relevance Score: 0.727
   Context: created the sacred band of carthage 2 500 elite citizen soldiers unlike our usual mercenary armies these were wealthy carthaginians fighting


Ask a historical question (or 'quit'):  quit


Exiting historical system.
